In [1]:
import numpy as np
import pandas as pd
from numpy import array
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [2]:
population_df = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/AnalyseProject/world_population.csv', index_col='Country Code')
meta_df = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/AnalyseProject/metadata.csv', index_col='Country Code')

In [3]:
population_df.head()

,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,...,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017
Country Code,,,,,,,,,,,,,,,,,,,,,
ABW,54211.0,55438.0,56225.0,56695.0,57032.0,57360.0,57715.0,58055.0,58386.0,58726.0,...,101353.0,101453.0,101669.0,102053.0,102577.0,103187.0,103795.0,104341.0,104822.0,105264.0
AFG,8996351.0,9166764.0,9345868.0,9533954.0,9731361.0,9938414.0,10152331.0,10372630.0,10604346.0,10854428.0,...,27294031.0,28004331.0,28803167.0,29708599.0,30696958.0,31731688.0,32758020.0,33736494.0,34656032.0,35530081.0
AGO,5643182.0,5753024.0,5866061.0,5980417.0,6093321.0,6203299.0,6309770.0,6414995.0,6523791.0,6642632.0,...,21759420.0,22549547.0,23369131.0,24218565.0,25096150.0,25998340.0,26920466.0,27859305.0,28813463.0,29784193.0
ALB,1608800.0,1659800.0,1711319.0,1762621.0,1814135.0,1864791.0,1914573.0,1965598.0,2022272.0,2081695.0,...,2947314.0,2927519.0,2913021.0,2905195.0,2900401.0,2895092.0,2889104.0,2880703.0,2876101.0,2873457.0
AND,13411.0,14375.0,15370.0,16412.0,17469.0,18549.0,19647.0,20758.0,21890.0,23058.0,...,83861.0,84462.0,84449.0,83751.0,82431.0,80788.0,79223.0,78014.0,77281.0,76965.0


In [4]:
meta_df.head()

,Region,Income Group,Special Notes
Country Code,,,
ABW,Latin America & Caribbean,High income,Mining is included in agriculture\r\r\r\nElect...
AFG,South Asia,Low income,Fiscal year end: March 20; reporting period fo...
AGO,Sub-Saharan Africa,Lower middle income,NaN
ALB,Europe & Central Asia,Upper middle income,NaN
AND,Europe & Central Asia,High income,WB-3 code changed from ADO to AND to align wit...


In [15]:
def get_population_by_income_group(income_group_name='Low income'):
    """
    Returns a 2D numpy array containing the year and total population
    for all countries within a given income group.

    Parameters
    ----------
    income_group_name : str, optional
        The name of the income group to filter by.
        Default is 'Low income'.

    Returns
    -------
    np.ndarray
        A 2D numpy array of shape (n, 2) where the first column contains
        the year and the second column contains the total population,
        both of type np.int64.

    Raises
    ------
    ValueError
        If the specified income_group_name does not exist in the dataset.
    """
    # Validate income group
    if income_group_name not in meta_df['Income Group'].values:
        raise ValueError(f"Income group '{income_group_name}' does not exist.")

    # Filter metadata to get country codes for the given income group
    country_codes = meta_df[meta_df['Income Group'] == income_group_name].index

    # Filter population data to only include countries in the income group
    filtered_df = population_df[population_df.index.isin(country_codes)]

    # Get year columns only
    year_cols = [col for col in population_df.columns if str(col).isdigit()]

    # Sum population across all countries for each year
    population_by_year = filtered_df[year_cols].sum(axis=0)

    # Build the 2D array of [year, population]
    result = array(
        [[int(year), int(pop)] for year, pop in population_by_year.items()],
        dtype=np.int64
    )

    return result

In [16]:
data = get_population_by_income_group('High income')
data

array([[      1960,  769889923],
       [      1961,  781225329],
       [      1962,  791207437],
       [      1963,  801108277],
       [      1964,  810900987],
       [      1965,  820309686],
       [      1966,  829088382],
       [      1967,  837479954],
       [      1968,  844905494],
       [      1969,  854059674],
       [      1970,  862276721],
       [      1971,  871169187],
       [      1972,  880246152],
       [      1973,  888486025],
       [      1974,  897803169],
       [      1975,  906573084],
       [      1976,  913843314],
       [      1977,  921330504],
       [      1978,  928906293],
       [      1979,  936836246],
       [      1980,  944587066],
       [      1981,  952368316],
       [      1982,  959759971],
       [      1983,  966754949],
       [      1984,  973423742],
       [      1985,  980143630],
       [      1986,  987194728],
       [      1987,  994242786],
       [      1988, 1001421456],
       [      1989, 1009036892],
       [  

In [17]:
help(KFold)

Help on class KFold in module sklearn.model_selection._split:

class KFold(_UnsupportedGroupCVMixin, _BaseKFold)
 |  KFold(n_splits=5, *, shuffle=False, random_state=None)
 |
 |  K-Fold cross-validator.
 |
 |  Provides train/test indices to split data in train/test sets. Split
 |  dataset into k consecutive folds (without shuffling by default).
 |
 |  Each fold is then used once as a validation while the k - 1 remaining
 |  folds form the training set.
 |
 |  Read more in the :ref:`User Guide <k_fold>`.
 |
 |  For visualisation of cross-validation behaviour and
 |  comparison between common scikit-learn split methods
 |  refer to :ref:`sphx_glr_auto_examples_model_selection_plot_cv_indices.py`
 |
 |  Parameters
 |  ----------
 |  n_splits : int, default=5
 |      Number of folds. Must be at least 2.
 |
 |      .. versionchanged:: 0.22
 |          ``n_splits`` default value changed from 3 to 5.
 |
 |  shuffle : bool, default=False
 |      Whether to shuffle the data before splitting int

In [18]:
def sklearn_kfold_split(data,K):
    """
    Splits a 2D numpy array into K folds using sklearn's KFold class.

    Parameters
    ----------
    data : np.ndarray
        A 2D numpy array to be split into K folds.
    K : int
        The number of folds to split the data into.

    Returns
    -------
    list
        A list of K tuples where each tuple contains:
        - train_indices : list of indices for the training set
        - test_indices  : list of indices for the testing set
    """
    kf = KFold(n_splits=K, shuffle=False)

    data_indices = [
        (list(train_indices), list(test_indices))
        for train_indices, test_indices in kf.split(data)
    ]

    return data_indices

In [19]:
sklearn_kfold_split(data,4)

[([np.int64(15),
   np.int64(16),
   np.int64(17),
   np.int64(18),
   np.int64(19),
   np.int64(20),
   np.int64(21),
   np.int64(22),
   np.int64(23),
   np.int64(24),
   np.int64(25),
   np.int64(26),
   np.int64(27),
   np.int64(28),
   np.int64(29),
   np.int64(30),
   np.int64(31),
   np.int64(32),
   np.int64(33),
   np.int64(34),
   np.int64(35),
   np.int64(36),
   np.int64(37),
   np.int64(38),
   np.int64(39),
   np.int64(40),
   np.int64(41),
   np.int64(42),
   np.int64(43),
   np.int64(44),
   np.int64(45),
   np.int64(46),
   np.int64(47),
   np.int64(48),
   np.int64(49),
   np.int64(50),
   np.int64(51),
   np.int64(52),
   np.int64(53),
   np.int64(54),
   np.int64(55),
   np.int64(56),
   np.int64(57)],
  [np.int64(0),
   np.int64(1),
   np.int64(2),
   np.int64(3),
   np.int64(4),
   np.int64(5),
   np.int64(6),
   np.int64(7),
   np.int64(8),
   np.int64(9),
   np.int64(10),
   np.int64(11),
   np.int64(12),
   np.int64(13),
   np.int64(14)]),
 ([np.int64(0),
   np

In [23]:
def best_k_model(data, data_indices):
    """
    Trains a RandomForestRegressor on each K-fold split and returns the
    model with the highest mean squared error on its testing set.

    Parameters
    ----------
    data : np.ndarray
        A 2D numpy array where the first column contains the year and
        the second column contains the population.
    data_indices : list
        A list of K tuples containing (train_indices, test_indices)
        for each fold.

    Returns
    -------
    RandomForestRegressor
        The trained model that obtained the highest MSE on its
        testing set across all K splits.
    """
    best_model = None
    best_mse = -np.inf

    for train_indices, test_indices in data_indices:
        # Split data into training and testing sets
        X_train, y_train = data[train_indices, 0], data[train_indices, 1]
        X_test, y_test   = data[test_indices, 0],  data[test_indices, 1]

        # Train the model
        rf = RandomForestRegressor(random_state=42)
        rf.fit(X_train.reshape(-1, 1), y_train)

        # Evaluate the model
        y_pred = rf.predict(X_test.reshape(-1, 1))
        mse = mean_squared_error(y_test, y_pred)

        # Keep track of the best model
        if mse > best_mse:
            best_mse = mse
            best_model = rf

    return best_model

In [24]:
data = get_population_by_income_group('High income')
data_indices = sklearn_kfold_split(data,5)

best_model = best_k_model(data,data_indices)
best_model.predict([[1960]])

array([8.85170916e+08])